<a href="https://colab.research.google.com/github/arildbn/bban4040/blob/main/martra-notebooks/3-3_data-analyst-agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div>
    <h1>Large Language Models Projects</a></h1>
    <h3>Apply and Implement Strategies for Large Language Models</h3>
    <h2>3.3-Create a Data Analyst Assistant using a LLM Agent.</h2>
    <p>by <b>Pere Martra</b></p>
</div>

# Create an LLMAgent with LangChain

We are going to create an Agent With LangChain that using the OpenAI API, will be able to analyze the data contained in an Excel file.

It will be able to find relationships between variables, clean the data, search for a model, and execute it to make future predictions.

In summary, it will act as a Data Scientist Assistant, helping us in our day-to-day tasks.

## Intalling and importing libraries

In [ ]:
# === Colab dependency guard (bban4040) ===
# The langchain / langsmith stack upgrades transitive packages (requests,
# opentelemetry-*) past the exact versions Colab's preinstalled google
# packages pin (google-colab, google-adk, the otlp/gcp exporters), which
# prints noisy "pip's dependency resolver ... is incompatible" errors.
# We pin those families to the versions already installed so the installs
# below leave them untouched. PIP_CONSTRAINT is honored by every %pip call
# in this kernel. Harmless off Colab (nothing matches / nothing to pin).
import os, tempfile
from importlib import metadata

_keep = []
for _dist in metadata.distributions():
    _name = (_dist.metadata.get("Name") or "").strip()
    if not _name:
        continue
    if _name.lower() == "requests" or _name.lower().startswith("opentelemetry"):
        _keep.append(f"{_name}=={_dist.version}")

if _keep:
    _con = os.path.join(tempfile.gettempdir(), "bban4040_pip_constraints.txt")
    with open(_con, "w") as _f:
        _f.write("\n".join(sorted(set(_keep))) + "\n")
    os.environ["PIP_CONSTRAINT"] = _con
    print(f"Pinned {len(_keep)} package(s) to avoid Colab pip resolver conflicts.")


In [ ]:
# Install the LangChain libraries needed for the pandas DataFrame agent.
# create_pandas_dataframe_agent lives in langchain-experimental, which is frozen
# on the langchain 0.3.x line (it requires langchain-core<0.4). We pin the whole
# langchain family to 0.3.x so the versions stay mutually compatible — mixing it
# with the 1.x packages (langchain-core 1.x / langchain_classic) is what caused
# the resolver conflicts and the "No module named 'langchain_classic'" error.
#
# Colab also preinstalls langgraph 1.x, which demands langchain-core 1.x and is
# not used by this notebook; remove it so pip's resolver stays clean once we
# pin core back to 0.3.x.
%pip uninstall -y -q langgraph langgraph-prebuilt langgraph-sdk langgraph-checkpoint langgraph-checkpoint-sqlite
%pip install -q \
  "langchain>=0.3,<1.0" \
  "langchain-core>=0.3.66,<1.0" \
  "langchain-experimental>=0.3,<1.0" \
  "langchain-openai>=0.2,<1.0" \
  tabulate

In [ ]:
# === portable-setup (bban4040) ===
# Secrets resolve from Colab "Secrets" (userdata) on Colab, or environment
# variables / a local .env file when running locally. Nothing is hardcoded.
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass


def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass
    value = os.environ.get(name, default)
    return value.strip() if isinstance(value, str) else value


_key = get_secret("OPENAI_API_KEY")
if _key:
    os.environ["OPENAI_API_KEY"] = _key


We use the **os** library to store Environ variables. Like OPENAI_API_KEY.

Get you OpenAI API  Key: https://platform.openai.com/

In [ ]:
import os
from getpass import getpass
os.environ["OPENAI_API_KEY"] = get_secret("OPENAI_API_KEY") or getpass("OpenAI API Key: ")

## Loading the Data

In [ ]:
# kagglehub is Kaggle's official downloader: it authenticates, caches, and
# extracts a dataset in a single call (cleaner than the kaggle CLI + manual unzip).
%pip install -q kagglehub

In [ ]:
import os
import kagglehub

# Resolve Kaggle credentials from Colab Secrets / env vars / .env (never
# hardcoded). kagglehub authenticates via KAGGLE_USERNAME + KAGGLE_KEY, or a
# ~/.kaggle/kaggle.json. Get a token at https://www.kaggle.com/settings ->
# "Create New Token".
for _name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
    _val = get_secret(_name)
    if _val:
        os.environ[_name] = _val

# Download (and cache) the real dataset from Kaggle, then expose its local
# folder. This raises a clear authentication error if credentials are missing:
# by design the notebook uses the real data and does NOT fall back to a sample.
dataset_dir = kagglehub.dataset_download("goyaladi/climate-insights-dataset")
print("Dataset downloaded to:", dataset_dir)

We use a Kaggle dataset: https://www.kaggle.com/datasets/goyaladi/climate-insights-dataset

The notebook downloads the **real** dataset directly from Kaggle (no synthetic data). You need Kaggle API credentials:

1. Go to https://www.kaggle.com/settings → **Create New Token** — this downloads a `kaggle.json` containing your username and key.
2. Provide them to the notebook in one of these ways:
   - **Colab (recommended):** add `KAGGLE_USERNAME` and `KAGGLE_KEY` as **Secrets** (the 🔑 icon in the left sidebar) and enable notebook access.
   - **Locally:** place `kaggle.json` at `~/.kaggle/kaggle.json` (`chmod 600`), or export `KAGGLE_USERNAME` / `KAGGLE_KEY` as environment variables.

If credentials are missing, the download cell raises a clear authentication error — it does **not** fall back to fake data.

In [ ]:
import os
import glob
import pandas as pd

# Locate the climate CSV inside the downloaded dataset. The archive ships
# climate_change_data.csv (possibly nested in a subfolder), so search the tree.
_csvs = glob.glob(os.path.join(dataset_dir, "**", "*.csv"), recursive=True)
if not _csvs:
    raise FileNotFoundError(
        f"No CSV found under {dataset_dir} - check the Kaggle download cell above."
    )
csv_file = _csvs[0]
print(f"Loading dataset from {csv_file}")
document = pd.read_csv(csv_file)
print(f"Loaded {len(document):,} rows x {document.shape[1]} columns")

In [ ]:
document.head(5)

In [ ]:
#If you want to use your own CSV just execute this Cell
#from google.colab import files

#def load_csv_file():
#  """Loads a CSV file into a Pandas dataframe."""

#  uploaded_file = files.upload()
#  file_path = next(iter(uploaded_file))
#  document = pd.read_csv(file_path)
#  return document

#if __name__ == "__main__":
#  document = load_csv_file()
#  print(document)

# Creating the Agent
This is the easiest Agent we can create with LangChain, we only need to import the **create_pandas_dataframe_agent**.

Time to create our little assistant, and we need only a call.

We let **OpenAI** decide which model to use. However, we specify a **temperature** value of 0 to its parameter, so that it is not imaginative. This is much better when we want the model to be able to give commands to the different libraries it can use.



In [ ]:
# AgentType comes from langchain (0.3.x), not langchain_classic (which is a
# 1.x-only package and incompatible with langchain-experimental's 0.3.x pin).
from langchain.agents.agent_types import AgentType
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAI

In [ ]:
# handle_parsing_errors must be passed via agent_executor_kwargs. As a bare
# kwarg langchain-experimental silently drops it (it warns "no longer
# supported"), leaving handle_parsing_errors=False — which let the ReAct output
# parser crash with OutputParserException when the model returned a plain-prose
# final answer. Routed here it reaches the AgentExecutor, so it retries instead
# of crashing. (verbose IS a recognized top-level arg, so it stays up top —
# putting it in agent_executor_kwargs too would be a duplicate.)
sm_ds_OAI = create_pandas_dataframe_agent(
    OpenAI(temperature=0),
    document,
    verbose=True,
    allow_dangerous_code=True,  # required since langchain-experimental 0.0.15+
    agent_executor_kwargs={"handle_parsing_errors": True},
)

In [ ]:
sm_ds_Chat = create_pandas_dataframe_agent(
    ChatOpenAI(temperature=0),
    document,
    verbose=True,
    allow_dangerous_code=True,  # required since langchain-experimental 0.0.15+
    agent_executor_kwargs={"handle_parsing_errors": True},  # see note on sm_ds_OAI
)

We are going to test 2 different models. The recommendation is use the one created withg the class OpenAI, but judge it by yourself.

## First Question.

In [ ]:
sm_ds_OAI.invoke("Analyze this data, and write a brief explanation around 100 words.")

In [ ]:
document.info()

The description of the data made by the Agent is acurated.

In [ ]:
sm_ds_Chat.invoke("""Analyze this data, and write a brief explanation around 100 words. """)

The second Agent is unable to solve this question.

## Second Question.

In [ ]:
sm_ds_OAI.run("Do you think is possible to forecast the temperature?")


The model thinks that is possible to forecast the temperature, but difficult because the weak correlation between variables.

I don't now why the model decided to create a graphic bar.

In [ ]:
sm_ds_Chat.run("Do you think is possible to forecast the temperature?")

The second Agent, created with ChatOpenAI has a different opinion.

## Third question

In [ ]:
sm_ds_OAI.run("""
Can you create a line graph containing the anual average co2 emissions over the years?
""")


## Fourth question

In [ ]:
sm_ds_OAI.run("""
Create a line graph with seaborn containing the anual average co2 emissions in Portugal over the years.
""")

## Last Question.

In [ ]:
sm_ds_OAI.invoke("""Select a forecasting model to forecast the temperature.
Use this kind of model to forecast the average temperature
for year in Port Maryberg in Malta for the next 5 years.
Write the temperatures forecasted in a table.""")

# Conclusions

This is one of the most powerful and, at the same time, easiest to use agents. We have seen how with just a few lines of code, we had an agent capable of following our instructions to analyze, clean, and generate charts from our data. Not only that, but it has also been able to draw conclusions and even decide which algorithm was best for forecasting the data.

The world of agents is just beginning, and many players are entering the field, such as Hugging Face, Microsoft, or Google. Their capabilities are not only growing with new tools but also with new language models.

**It's a revolution that we cannot afford to miss and will change many things.**